# Plan B (Revised) — Generate Training Data + Fine-tune

**Architecture:**
- Speakers: neuroscientist + philosopher (no names, no personal info)
- Topics: research or technology focused
- Augment **missing** perspectives across engineering, clinical, robotics, environment/climate, philosophy, aerial systems
- **RAG (Chroma on Drive):** only for missing **philosophy**, **robotics**, and **aerial systems**
- **Parametric (no RAG):** engineering, clinical, environment/climate
- **Train/eval alignment:** JSONL `user` messages are the **full structured prompt** (transcript + missing-domain instructions + optional retrieval blocks), identical to what the label generator and holdout eval use — **not** the raw transcript alone. Skipped rows have no missing domains to augment.
- **Grounding / relevance:** RAG uses topic-aware queries + distance filtering; prompts require claims to follow retrieved text and discourage off-topic stretching.
- **Outputs:** JSON/JSONL/eval artifacts are written under `Pivot/data/revised/` on Drive (`DATA_REVISED_DIR` after mount).
- 40 topic examples; JSONL is split into **train** + **validation**; fine-tuning uses `validation_file`
- After training, the notebook picks the **checkpoint with lowest validation loss** for evaluation


In [1]:
# Cell 1: Install dependencies
!pip install openai --quiet

In [17]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opente

In [2]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
# Generated artifacts from this notebook (JSON, JSONL, eval, checkpoints metadata)
DATA_REVISED_DIR = '/content/drive/MyDrive/Pivot/data/revised'
os.makedirs(DATA_REVISED_DIR, exist_ok=True)
print(f"Output directory ready: {DATA_REVISED_DIR}")


Mounted at /content/drive


In [ ]:
# Cell 3: Set up OpenAI client
from openai import OpenAI

OPENAI_API_KEY = ""  # paste your key here
oai_client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI ready!")

OpenAI ready!


In [18]:
# Cell 3b: Load ChromaDB and sentence transformer
import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_PATH = '/content/drive/MyDrive/Pivot/chromadb'
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection("domain_knowledge")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f"ChromaDB loaded! {collection.count()} documents available.")
print("Embedder ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB loaded! 981 documents available.
Embedder ready!


In [19]:
# Cell 4: OpenAI call helper
def call_openai(system_prompt, user_prompt, max_tokens=1200):
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.8,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

print("Helper ready!")

Helper ready!


In [20]:
# Cell 5: Define all 50 topics
# 40 for training, 10 held out for evaluation
# Speakers: always neuroscientist + philosopher
# Each topic is a research or technology subject

ALL_TOPICS = [
    # --- TRAINING SET (40 topics) ---
    # Neuroscience + AI
    "The use of brain-computer interfaces to restore motor function in paralyzed patients",
    "Whether AI can accurately model human consciousness and what that means for neuroscience",
    "The ethics of using neural decoding to read mental states without explicit consent",
    "How optogenetics could reshape our understanding of memory formation",
    "Whether neuroplasticity research supports the idea of cognitive enhancement through technology",
    "The implications of AI-assisted diagnosis of neurological disorders like Alzheimer's",
    "How deep brain stimulation raises questions about personal identity and autonomy",
    "Whether artificial neural networks are valid models of biological neural systems",
    "The role of neurofeedback in treating PTSD and anxiety disorders",
    "How connectome mapping could change our understanding of the self",

    # Research ethics + technology
    "Whether CRISPR gene editing in humans crosses an ethical line",
    "The tension between open science and protecting sensitive neurological data",
    "How psychedelic-assisted therapy challenges existing models of mental health treatment",
    "Whether AI should be used to predict mental health crises before they occur",
    "The ethics of animal experimentation in neuroscience research",
    "How wearable biosensors change the boundaries between health monitoring and surveillance",
    "Whether cognitive behavioral therapy delivered by AI can be as effective as human therapy",
    "The philosophical implications of discovering neurons responsible for specific emotions",
    "How sleep research and AI intersect in diagnosing sleep disorders",
    "Whether long-term space travel has irreversible effects on the human brain",

    # Emerging tech + society
    "The implications of AI systems that can predict human decision-making from brain scans",
    "Whether digital twins of the human brain are scientifically or ethically feasible",
    "How childhood screen exposure affects long-term neurological development",
    "The ethics of enhancing soldier cognition using neurostimulation in warfare",
    "Whether AI-generated art challenges existing theories of creativity and the mind",
    "How the study of neurological disorders in aging populations shapes research priorities",
    "Whether consciousness could exist in a sufficiently complex artificial system",
    "The role of mirror neurons in empathy and implications for social AI design",
    "How AI tools for protein folding are transforming neurodegenerative disease research",
    "Whether the human brain is fundamentally a prediction machine and what that means for AI",

    # Philosophy of mind + science
    "The hard problem of consciousness and whether neuroscience can ever fully resolve it",
    "Whether free will is compatible with deterministic neural processes",
    "How the placebo effect challenges the boundary between mind and body in medical research",
    "Whether language models challenge the uniqueness of human linguistic cognition",
    "The ethics of publishing research on cognitive differences across demographic groups",
    "Whether emotion is a biological phenomenon or a socially constructed experience",
    "How advances in neuroimaging are changing theories of personal identity over time",
    "Whether AI decision-making systems should incorporate models of human moral reasoning",
    "The implications of discovering a neural basis for religious or spiritual experiences",
    "Whether human memory is reliable enough to serve as evidence in legal contexts",

    # --- HOLDOUT SET (10 topics — used for evaluation only, never trained on) ---
    "The ethics of using AI to detect depression from social media activity",
    "Whether neuroethics should be a mandatory part of AI research training",
    "How brain organoids challenge our definition of what constitutes a living system",
    "Whether AI can be used to fairly allocate scarce neurosurgical resources",
    "The tension between reductionist neuroscience and holistic views of mental health",
    "Whether cognitive load theory should inform the design of AI interfaces",
    "How trauma research is reshaping our understanding of memory consolidation",
    "Whether there is a meaningful distinction between brain enhancement and brain repair",
    "The implications of AI systems that outperform humans in diagnosing rare diseases",
    "Whether the concept of mental illness is a scientific or cultural construct",
]

TRAINING_TOPICS = ALL_TOPICS[:40]
HOLDOUT_TOPICS = ALL_TOPICS[40:]

print(f"Training topics: {len(TRAINING_TOPICS)}")
print(f"Holdout topics: {len(HOLDOUT_TOPICS)}")

Training topics: 40
Holdout topics: 10


In [21]:
# Cell 6: Function to simulate a neuroscientist + philosopher conversation

def simulate_transcript(topic):
    system_prompt = """You are simulating a realistic academic conversation between a neuroscientist
and a philosopher. Do not use any real names or personal information — refer to them only as
'Neuroscientist' and 'Philosopher'. The conversation should:
- Be intellectually rigorous and reflect each speaker's academic background
- Show genuine tension, disagreement, and open questions
- Stay focused on research and technology implications
- Feel like a real academic discussion, not a scripted debate
- Have 10-12 turns total
Format: Speaker: [what they say]"""

    user_prompt = f"""Simulate a conversation between a Neuroscientist and a Philosopher
discussing the following research topic: '{topic}'

The neuroscientist should bring empirical, mechanistic thinking.
The philosopher should bring conceptual, ethical, and epistemological thinking.
Leave tensions unresolved — this is an ongoing academic dialogue."""

    return call_openai(system_prompt, user_prompt, max_tokens=900)

# Test the function
print(simulate_transcript(TRAINING_TOPICS[0]))

Neuroscientist: The advancements in brain-computer interfaces (BCIs) have been remarkable. Recent studies demonstrate that we can decode motor intentions from neural signals, allowing paralyzed patients to control prosthetic limbs with their thoughts. This mechanical understanding of the brain's role in movement opens up possibilities for restoring function.

Philosopher: While the empirical successes are indeed compelling, they raise fundamental questions about agency and identity. When a paralyzed individual moves a prosthetic limb using a BCI, to what extent is that action genuinely theirs? Are we merely hijacking neural pathways, or are we restoring a sense of self through these technologies?

Neuroscientist: I appreciate the philosophical considerations, but we have to ground this discussion in the efficacy of the technology. The objective is to provide functional outcomes for patients. If they can perform tasks they couldn't before, doesn't that enhance their autonomy, regardless

In [22]:
# Cell 7: Function to detect which perspectives are present in the transcript

def detect_present_perspectives(transcript):
    system_prompt = """You are an expert at identifying domain perspectives in academic conversations.
Analyze the transcript and identify which of these domains are already represented:
- neuroscience
- philosophy
- engineering
- clinical
- robotics
- environment/climate
- aerial systems

Return ONLY a comma-separated list of present domains using the exact labels above. Nothing else."""

    user_prompt = f"Transcript:\n{transcript}"
    result = call_openai(system_prompt, user_prompt, max_tokens=80)
    return [d.strip().lower() for d in result.split(',')]

# Test detect function using the transcript from Cell 6
test_transcript = simulate_transcript(TRAINING_TOPICS[0])
present = detect_present_perspectives(test_transcript)
print(f"Present perspectives: {present}")


Present perspectives: ['neuroscience', 'philosophy']


In [23]:
# Cell 8: Build augmentation prompt + generate missing perspectives
# RAG: philosophy, robotics, aerial systems (Chroma metadata `domain` must match CHROMA_DOMAIN values)
# Parametric: engineering, clinical, environment/climate
#
# Relevance: topic-aware retrieval query; weak chunks filtered by embedding distance.
# Grounding: strict SYSTEM_PROMPT + user rules — cite only when excerpts support the claim.

CORE_DOMAINS = ['engineering', 'clinical', 'robotics']
SUGGESTIVE_DOMAINS = ['environment/climate']
AUX_DOMAINS = ['philosophy', 'aerial systems']
RAG_DOMAINS = frozenset(['philosophy', 'robotics', 'aerial systems'])

CHROMA_DOMAIN = {
    'philosophy': 'philosophy',
    'robotics': 'robotics',
    'aerial systems': 'aerial systems',
}

# Retrieve more candidates, then keep closest chunks (tune RAG_MAX_DISTANCE / MARGIN if needed).
RAG_CANDIDATES = 8
RAG_MAX_CHUNKS = 3
RAG_MAX_DISTANCE = 1.05
RAG_DISTANCE_MARGIN = 0.18

SYSTEM_PROMPT = """You are a research assistant that identifies missing domain perspectives
in interdisciplinary academic conversations between a neuroscientist and a philosopher.
You add engineering, clinical, robotics, environment/climate (when relevant), philosophy,
and aerial systems lenses only when they are genuinely missing from the dialogue.

Grounding rules (strict):
- When source excerpts are provided, every factual claim tied to those excerpts must be
  directly supported by quoted or clearly paraphrased material. Cite the source file name.
- If an excerpt is only weakly related to the conversation topic, say so briefly and either
  limit claims to what the text actually supports or do not use that excerpt to justify broad claims.
- Never invent study results, statistics, or citations. For domains without excerpts, use careful
  general knowledge and do not add fake references.
- Stay on-topic: do not force a domain if the only connection would be superficial.
- Prefer one well-grounded paragraph over two vague ones."""

def _retrieval_query_text(transcript, domain, topic_hint=None):
    t = transcript.strip()
    if len(t) > 1800:
        t = t[:1800] + "\n[...]"
    if topic_hint:
        return (
            f"Research / discussion topic: {topic_hint}\n\n"
            f"Find {domain} ideas (methods, constraints, technologies, ethics) relevant to this dialogue.\n\n"
            f"Transcript:\n{t}"
        )
    return (
        f"Find {domain} ideas (methods, constraints, technologies, ethics) relevant to this academic dialogue.\n\n"
        f"Transcript:\n{t}"
    )

def retrieve_chunks(transcript, domain, topic_hint=None):
    """Return (chunks, sources) filtered by distance; may be empty if collection is empty."""
    chroma_key = CHROMA_DOMAIN.get(domain, domain)
    try:
        n_docs = collection.count()
    except Exception:
        n_docs = RAG_CANDIDATES
    if n_docs <= 0:
        return [], []
    n_q = min(RAG_CANDIDATES, n_docs)
    qtext = _retrieval_query_text(transcript, domain, topic_hint)
    query_embedding = embedder.encode(qtext[:8000]).tolist()
    try:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_q,
            where={"domain": chroma_key},
            include=["documents", "metadatas", "distances"],
        )
    except Exception:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_q,
            where={"domain": chroma_key},
        )
    chunks = results.get('documents', [[]])[0] or []
    metas = results.get('metadatas', [[]])[0] or []
    dists = results.get('distances', [[]])[0]
    sources = [m.get('source_file', m.get('source', 'unknown')) for m in metas]
    seen = set()
    paired = []
    fill = dists if dists is not None else [None] * len(chunks)
    for ch, src, di in zip(chunks, sources, fill):
        key = (ch[:120], src)
        if key in seen:
            continue
        seen.add(key)
        paired.append((ch, src, di))
    if not paired:
        return [], []
    if paired[0][2] is None:
        paired = paired[:RAG_MAX_CHUNKS]
        return [p[0] for p in paired], [p[1] for p in paired]
    paired.sort(key=lambda x: x[2])
    best_d = paired[0][2]
    limit = min(RAG_MAX_DISTANCE, best_d + RAG_DISTANCE_MARGIN)
    filtered = [(c, s) for c, s, d in paired if d <= limit]
    if not filtered:
        filtered = [(paired[0][0], paired[0][1])]
    filtered = filtered[:RAG_MAX_CHUNKS]
    return [x[0] for x in filtered], [x[1] for x in filtered]

def _missing_to_augment(present_perspectives):
    present = set(present_perspectives)
    missing = []
    for d in CORE_DOMAINS:
        if d not in present:
            missing.append(d)
    for d in SUGGESTIVE_DOMAINS:
        if d not in present:
            missing.append(d)
    for d in AUX_DOMAINS:
        if d not in present:
            missing.append(d)
    return missing

def build_augmentation_user_prompt(transcript, present_perspectives, topic_hint=None):
    """Returns (user_prompt, rag_context_section) or (None, '') if nothing to add."""
    missing = _missing_to_augment(present_perspectives)
    if not missing:
        return None, ''

    rag_context_section = ""
    for domain in missing:
        if domain in RAG_DOMAINS:
            chunks, sources = retrieve_chunks(transcript, domain, topic_hint=topic_hint)
            if chunks:
                rag_context_section += f"\n--- Retrieved knowledge for {domain} ---\n"
                for i, (chunk, source) in enumerate(zip(chunks, sources)):
                    rag_context_section += f"[Source {i+1}: {source}]\n{chunk[:450]}\n\n"

    missing_core = [d for d in missing if d in CORE_DOMAINS]
    missing_suggestive = [d for d in missing if d in SUGGESTIVE_DOMAINS]
    missing_aux = [d for d in missing if d in AUX_DOMAINS]

    lines = []
    if missing_core:
        lines.append(f"Missing core perspectives: {', '.join(missing_core)}")
    if missing_suggestive:
        lines.append(f"Suggestive perspectives (include only if clearly relevant to the topic): {', '.join(missing_suggestive)}")
    if missing_aux:
        lines.append(f"Additional missing lenses (substantive, on-topic only): {', '.join(missing_aux)}")
    missing_section = "\n".join(lines) + "\n"

    rag_block = (
        f"Retrieved excerpts (philosophy / robotics / aerial only). Use ONLY insofar as they clearly apply; otherwise say relevance is limited.:{rag_context_section}"
        if rag_context_section else ""
    )

    user_prompt = f"""Here is an academic conversation between a neuroscientist and a philosopher:

{transcript}

{missing_section}
{rag_block}

Instructions:
1. For each missing perspective you address, stay tightly connected to the actual debate.
2. If retrieved excerpts are off-topic or only tangential, state that and avoid stretching them into strong claims.
3. When you use an excerpt, tie claims to specific phrases or ideas from that excerpt and name the source file.
4. For domains without excerpts, use general knowledge — no fabricated citations or paper titles.
5. For suggestive perspectives, include only if clearly relevant; otherwise omit that section entirely.

Use this exact format for each missing perspective you include:

Missing perspective from [domain]:
Key insight: [one sentence]
Contribution: [2-3 paragraphs — retrieved sources only where clearly supported]
Follow-up questions: [2 questions this perspective raises]

---

Only include perspectives that are genuinely missing. Do not repeat what is already in the conversation."""
    return user_prompt, rag_context_section

def generate_missing_perspectives(transcript, present_perspectives, topic_hint=None):
    user_prompt, _rag = build_augmentation_user_prompt(transcript, present_perspectives, topic_hint=topic_hint)
    if user_prompt is None:
        return "All key perspectives are already represented in this conversation."
    return call_openai(SYSTEM_PROMPT, user_prompt, max_tokens=1200)


In [24]:
# Cell 9: Preview one example before generating all
import time

test_topic = TRAINING_TOPICS[0]
print(f"Topic: {test_topic}\n")

print("Simulating transcript...")
test_transcript = simulate_transcript(test_topic)
print(test_transcript)

print("\nDetecting present perspectives...")
present = detect_present_perspectives(test_transcript)
print(f"Present: {present}")

print("\nGenerating missing perspectives...")
output = generate_missing_perspectives(test_transcript, present, topic_hint=test_topic)
print(output)


Topic: The use of brain-computer interfaces to restore motor function in paralyzed patients

Simulating transcript...
Neuroscientist: The advances in brain-computer interfaces (BCIs) over the past decade have been remarkable. We can now translate neural signals from the motor cortex into real-time control of external devices, which can significantly restore movement for paralyzed patients. The mechanistic understanding of how neurons communicate provides a solid foundation for these technologies.

Philosopher: While I acknowledge the progress in the empirical realm, I can't help but wonder about the ethical implications of these technologies. For instance, how do we define 'restoration' in the context of motor function? If a BCI allows a patient to control a prosthetic limb, does that truly restore their agency, or does it merely simulate it?

Neuroscientist: That’s an intriguing point, but I would argue that the ability to move a prosthetic limb offers a level of agency that many para

In [25]:
# Cell 10: Generate all 40 training examples
import json
import time

training_examples = []
failed = []
chunk_counter = 0

for idx, topic in enumerate(TRAINING_TOPICS):
    print(f"\n[{idx+1}/40] {topic[:70]}...")

    try:
        print("  Simulating transcript...")
        transcript = simulate_transcript(topic)
        time.sleep(1)

        present = detect_present_perspectives(transcript)
        print(f"  Present perspectives: {present}")

        user_prompt, rag_context = build_augmentation_user_prompt(transcript, present, topic_hint=topic)
        print("  Generating missing perspectives...")
        if user_prompt is None:
            output = "All key perspectives are already represented in this conversation."
        else:
            output = call_openai(SYSTEM_PROMPT, user_prompt, max_tokens=1200)
        time.sleep(1)

        training_examples.append({
            "topic": topic,
            "present_perspectives": present,
            "input": transcript,
            "user_prompt": user_prompt,
            "rag_context": rag_context,
            "output": output,
        })
        print("  Done!")

    except Exception as e:
        print(f"  Failed: {e}")
        failed.append(topic)

    chunk_counter += 1
    if chunk_counter % 10 == 0:
        checkpoint_path = f"{DATA_REVISED_DIR}/revised_training_checkpoint.json"
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(training_examples, f, indent=2, ensure_ascii=False)
        print(f"\n  Checkpoint saved: {len(training_examples)} examples")

raw_path = f"{DATA_REVISED_DIR}/revised_training_examples.json"
with open(raw_path, 'w', encoding='utf-8') as f:
    json.dump(training_examples, f, indent=2, ensure_ascii=False)

print("\n--- Done ---")
print(f"Training examples generated: {len(training_examples)}")
print(f"Failed: {len(failed)}")



[1/40] The use of brain-computer interfaces to restore motor function in para...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy', 'clinical', 'robotics']
  ✍️ Generating missing perspectives...
  ✅ Done!

[2/40] Whether AI can accurately model human consciousness and what that mean...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy']
  ✍️ Generating missing perspectives...
  ✅ Done!

[3/40] The ethics of using neural decoding to read mental states without expl...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy', 'clinical']
  ✍️ Generating missing perspectives...
  ✅ Done!

[4/40] How optogenetics could reshape our understanding of memory formation...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy', 'clinical']
  ✍️ Generating missing perspectives...
  ✅ Done!

[5/40] Whether neuroplasticity research supports the idea of cognitive enhanc

In [26]:
# Cell 11: Generate 10 holdout examples (for evaluation only — never trained on)

holdout_examples = []
holdout_failed = []

for idx, topic in enumerate(HOLDOUT_TOPICS):
    print(f"\n[{idx+1}/10] {topic[:70]}...")

    try:
        print("  Simulating transcript...")
        transcript = simulate_transcript(topic)
        time.sleep(1)

        present = detect_present_perspectives(transcript)
        print(f"  Present perspectives: {present}")

        user_prompt, rag_context = build_augmentation_user_prompt(transcript, present, topic_hint=topic)
        print("  Generating missing perspectives...")
        if user_prompt is None:
            output = "All key perspectives are already represented in this conversation."
        else:
            output = call_openai(SYSTEM_PROMPT, user_prompt, max_tokens=1200)
        time.sleep(1)

        holdout_examples.append({
            "topic": topic,
            "present_perspectives": present,
            "input": transcript,
            "user_prompt": user_prompt,
            "rag_context": rag_context,
            "output": output,
        })
        print("  Done!")

    except Exception as e:
        print(f"  Failed: {e}")
        holdout_failed.append(topic)

holdout_path = f"{DATA_REVISED_DIR}/revised_holdout_examples.json"
with open(holdout_path, 'w', encoding='utf-8') as f:
    json.dump(holdout_examples, f, indent=2, ensure_ascii=False)

print("\n--- Done ---")
print(f"Holdout examples saved: {len(holdout_examples)}")
print(f"Saved to: {holdout_path}")



[1/10] The ethics of using AI to detect depression from social media activity...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy', 'clinical']
  ✍️ Generating missing perspectives...
  ✅ Done!

[2/10] Whether neuroethics should be a mandatory part of AI research training...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy']
  ✍️ Generating missing perspectives...
  ✅ Done!

[3/10] How brain organoids challenge our definition of what constitutes a liv...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy']
  ✍️ Generating missing perspectives...
  ✅ Done!

[4/10] Whether AI can be used to fairly allocate scarce neurosurgical resourc...
  💬 Simulating transcript...
  👁️ Present perspectives: ['neuroscience', 'philosophy', 'clinical']
  ✍️ Generating missing perspectives...
  ✅ Done!

[5/10] The tension between reductionist neuroscience and holistic views of me...
  💬 Simulating tra

In [27]:
# Cell 12: Convert training examples to JSONL (train / validation split)
#
# Design: each JSONL user message == build_augmentation_user_prompt(...) (same as teacher + eval).
# Do not switch to example['input'] only; that would mismatch eval and hurt fine-tuning.
import json
import random

jsonl_train_path = f"{DATA_REVISED_DIR}/revised_finetune_train.jsonl"
jsonl_val_path = f"{DATA_REVISED_DIR}/revised_finetune_val.jsonl"
jsonl_full_path = f"{DATA_REVISED_DIR}/revised_finetune_all.jsonl"  # optional backup

VAL_COUNT = 8
rng = random.Random(42)
indices = list(range(len(training_examples)))
rng.shuffle(indices)
val_indices = set(indices[:VAL_COUNT])

finetune_rows = []
for i, example in enumerate(training_examples):
    up = example.get('user_prompt')
    if up is None:
        present = example.get('present_perspectives', [])
        up, _ = build_augmentation_user_prompt(example['input'], present, topic_hint=example.get('topic'))
    if up is None:
        continue
    finetune_rows.append({
        "i": i,
        "is_val": i in val_indices,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": up},
            {"role": "assistant", "content": example['output']},
        ],
    })

train_rows = [r for r in finetune_rows if not r['is_val']]
val_rows = [r for r in finetune_rows if r['is_val']]

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            rec = {"messages": r["messages"]}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

write_jsonl(jsonl_train_path, train_rows)
write_jsonl(jsonl_val_path, val_rows)
write_jsonl(jsonl_full_path, train_rows + val_rows)

split_meta = {
    "val_count": len(val_rows),
    "train_count": len(train_rows),
    "skipped_no_missing": len(training_examples) - len(finetune_rows),
    "val_indices": sorted(val_indices),
}
with open(f"{DATA_REVISED_DIR}/revised_finetune_split.json", 'w', encoding='utf-8') as f:
    json.dump(split_meta, f, indent=2)

print(f"Train JSONL: {jsonl_train_path} ({len(train_rows)} rows)")
print(f"Val JSONL:   {jsonl_val_path} ({len(val_rows)} rows)")
print(f"Skipped (no missing domains to augment): {split_meta['skipped_no_missing']}")


✅ JSONL saved: /content/drive/MyDrive/Pivot/data/revised_finetune.jsonl
Total records: 40


In [28]:
# Cell 13: Validate train + val JSONL before uploading to OpenAI
import json

def validate_jsonl(path, label):
    errors = []
    valid_count = 0
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                record = json.loads(line)
                assert 'messages' in record
                assert len(record['messages']) == 3
                assert record['messages'][0]['role'] == 'system'
                assert record['messages'][1]['role'] == 'user'
                assert record['messages'][2]['role'] == 'assistant'
                assert len(record['messages'][1]['content']) > 100
                assert len(record['messages'][2]['content']) > 50
                valid_count += 1
            except Exception as e:
                errors.append(f"{label} line {i+1}: {e}")
    return valid_count, errors

vc_t, err_t = validate_jsonl(jsonl_train_path, 'train')
vc_v, err_v = validate_jsonl(jsonl_val_path, 'val')
print(f"Train valid records: {vc_t}")
print(f"Val valid records: {vc_v}")
for e in err_t + err_v:
    print(f"  ERROR: {e}")
if not err_t and not err_v:
    print("All records valid — ready to upload!")


Valid records: 40
✅ All records valid — ready to upload!


In [29]:
# Cell 14: Upload train + validation JSONL to OpenAI

print("Uploading training file...")
with open(jsonl_train_path, 'rb') as f:
    tr = oai_client.files.create(file=f, purpose='fine-tune')
TRAIN_FILE_ID = tr.id
print(f"Train file ID: {TRAIN_FILE_ID}")

print("Uploading validation file...")
with open(jsonl_val_path, 'rb') as f:
    va = oai_client.files.create(file=f, purpose='fine-tune')
VAL_FILE_ID = va.id
print(f"Validation file ID: {VAL_FILE_ID}")


Uploading training file to OpenAI...
✅ File uploaded!
File ID: file-MGECdB7D2Vi2VhuwbucPeM


In [30]:
# Cell 15: Launch fine-tuning job (validation metrics enabled)

print("Launching fine-tuning job...")

job = oai_client.fine_tuning.jobs.create(
    training_file=TRAIN_FILE_ID,
    validation_file=VAL_FILE_ID,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={
        "n_epochs": 3,
    },
)

JOB_ID = job.id
print("Fine-tuning job launched!")
print(f"Job ID: {JOB_ID}")
print(f"Status: {job.status}")
print("\nTraining takes 20–60+ minutes. Run Cell 16 to check progress, then pick the best checkpoint.")


Launching fine-tuning job...
✅ Fine-tuning job launched!
Job ID: ftjob-A2TNXauA5Y2PzkUljbYahpd1
Status: validating_files

⏳ Training takes 20-60 minutes. Run Cell 16 to check progress.


In [32]:
# Cell 16: Job status + select checkpoint with lowest validation loss

job_status = oai_client.fine_tuning.jobs.retrieve(JOB_ID)
print(f"Status: {job_status.status}")

FINE_TUNED_MODEL = None
BEST_CHECKPOINT = None
CHECKPOINT_METRICS = None

if job_status.status == 'succeeded':
    # Default model (last epoch)
    FINE_TUNED_MODEL = job_status.fine_tuned_model
    print(f"\nJob finished. Default fine_tuned_model: {FINE_TUNED_MODEL}")

    all_ckpts = []
    after = None
    while True:
        page = oai_client.fine_tuning.jobs.checkpoints.list(fine_tuning_job_id=JOB_ID, after=after, limit=20)
        if not page.data:
            break
        all_ckpts.extend(page.data)
        if len(page.data) < 20:
            break
        after = page.data[-1].id

    def val_score(ckpt):
        m = getattr(ckpt, 'metrics', None)
        if m is None:
            return None
        if isinstance(m, dict):
            return m.get('full_valid_loss') or m.get('valid_loss')
        return getattr(m, 'full_valid_loss', None) or getattr(m, 'valid_loss', None)

    scored = [(c, val_score(c)) for c in all_ckpts]
    scored_w_val = [(c, v) for c, v in scored if v is not None]
    if scored_w_val:
        best_ckpt, best_val = min(scored_w_val, key=lambda x: x[1])
        BEST_CHECKPOINT = best_ckpt.fine_tuned_model_checkpoint
        CHECKPOINT_METRICS = {"best_valid_loss": best_val, "step_number": getattr(best_ckpt, 'step_number', None)}
        FINE_TUNED_MODEL = BEST_CHECKPOINT
        print(f"\nBest checkpoint (lowest validation loss): {BEST_CHECKPOINT}")
        print(f"Validation loss: {best_val}")
    else:
        print("\nNo per-checkpoint validation loss in API response; using job fine_tuned_model.")
        print("Check Cell 16b events / dashboard for curves.")

elif job_status.status == 'failed':
    print(f"\nFailed: {job_status.error}")
else:
    print("\nStill training... rerun this cell later.")

events = oai_client.fine_tuning.jobs.list_events(JOB_ID, limit=5)
print("\nRecent events:")
for event in events.data:
    print(f"  [{event.level}] {event.message}")


Status: succeeded

✅ Fine-tuning complete!
Fine-tuned model ID: ft:gpt-4o-mini-2024-07-18:duke-trust-lab::DQdbeckt

Recent events:
  [info] The job has successfully completed
  [info] Usage policy evaluations completed, model is now enabled for sampling
  [info] Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:duke-trust-lab::DQdbeckt passed.
  [info] Evaluating model against our usage policies
  [info] New fine-tuned model created


In [33]:
# Cell 16b: View training and validation loss from job events

events = oai_client.fine_tuning.jobs.list_events(JOB_ID, limit=100)

print(f"{'Step':<10} {'Train Loss':<15} {'Val Loss':<15}")
print("-" * 40)

for event in reversed(events.data):
    if not event.data:
        continue
    d = event.data if isinstance(event.data, dict) else {}
    if 'train_loss' not in d and 'valid_loss' not in d:
        continue
    step = d.get('step', '-')
    train_loss = d.get('train_loss', '-')
    vl = d.get('valid_loss', '-')
    if isinstance(train_loss, float):
        train_loss = round(train_loss, 4)
    if isinstance(vl, float):
        vl = round(vl, 4)
    print(f"{str(step):<10} {str(train_loss):<15} {str(vl):<15}")


Step       Train Loss      Val Loss       
----------------------------------------
78         0.6645          -              
79         0.7928          -              
80         0.6917          -              
81         0.6062          -              
82         0.7885          -              
83         0.4817          -              
84         0.6615          -              
85         0.6216          -              
86         0.6413          -              
87         0.6111          -              
88         0.6691          -              
89         0.739           -              
90         0.5939          -              
91         0.5462          -              
92         0.6503          -              
93         0.6765          -              
94         0.5632          -              
95         0.6107          -              
96         0.6953          -              
97         0.582           -              
98         0.5162          -              
99         0.

In [37]:
# Cell 17: Evaluate best checkpoint on holdout (same user prompt as training)
import json

assert FINE_TUNED_MODEL is not None, "Run Cell 16 after job succeeded"

holdout_path = f"{DATA_REVISED_DIR}/revised_holdout_examples.json"
with open(holdout_path, 'r', encoding='utf-8') as f:
    holdout_examples = json.load(f)

evaluation_results = []

def _print_eval_example(idx, total, r, show_retrieved=True, retrieved_max=2500):
    """Print one evaluation record to stdout."""
    print("\n" + "#" * 72)
    print(f"# Example {idx + 1} / {total}")
    print(f"# Topic: {r['topic']}")
    print("#" * 72)
    print(f"\nPresent perspectives (re-detected): {r.get('present_perspectives', [])}")
    rc = r.get("retrieved_context") or ""
    if show_retrieved and rc.strip():
        body = rc if len(rc) <= retrieved_max else (rc[:retrieved_max] + "\n... [truncated for console; full text in JSON file]")
        print(f"\n--- Retrieved context ---\n{body}")
    print(f"\n--- Ground truth (teacher) ---\n{r.get('ground_truth', '')}")
    print(f"\n--- Model prediction ---\n{r.get('model_prediction', '')}")
    print("\n" + "-" * 72)

for idx, example in enumerate(holdout_examples):
    print(f"\n[{idx+1}/{len(holdout_examples)}] Generating: {example['topic'][:70]}...")
    present = detect_present_perspectives(example['input'])
    user_prompt, rag_context = build_augmentation_user_prompt(example['input'], present, topic_hint=example.get('topic'))
    if user_prompt is None:
        user_prompt = (
            "The dialogue already appears to cover the target lenses. "
            "Reply briefly that no major missing perspectives were identified."
        )
    response = oai_client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=1200,
    )
    prediction = response.choices[0].message.content.strip()
    row = {
        "topic": example['topic'],
        "input": example['input'],
        "present_perspectives": present,
        "retrieved_context": rag_context,
        "ground_truth": example['output'],
        "model_prediction": prediction,
    }
    evaluation_results.append(row)
    _print_eval_example(idx, len(holdout_examples), row)

eval_path = f"{DATA_REVISED_DIR}/evaluation_results.json"
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 72)
print(f"Saved {len(evaluation_results)} examples to: {eval_path}")
print("Next: run Cell 17b (next cell) to print from JSON without API calls; use the side-by-side review cell after that.")




[1/10] The ethics of using AI to detect depression from social medi...
  ✅ Prediction generated

[2/10] Whether neuroethics should be a mandatory part of AI researc...
  ✅ Prediction generated

[3/10] How brain organoids challenge our definition of what constit...
  ✅ Prediction generated

[4/10] Whether AI can be used to fairly allocate scarce neurosurgic...
  ✅ Prediction generated

[5/10] The tension between reductionist neuroscience and holistic v...
  ✅ Prediction generated

[6/10] Whether cognitive load theory should inform the design of AI...
  ✅ Prediction generated

[7/10] How trauma research is reshaping our understanding of memory...
  ✅ Prediction generated

[8/10] Whether there is a meaningful distinction between brain enha...
  ✅ Prediction generated

[9/10] The implications of AI systems that outperform humans in dia...
  ✅ Prediction generated

[10/10] Whether the concept of mental illness is a scientific or cul...
  ✅ Prediction generated

✅ Evaluation results saved t

In [ ]:
# Cell 17b (optional): Print evaluation results from saved JSON (no API calls)
import json

eval_path = f"{DATA_REVISED_DIR}/evaluation_results.json"
with open(eval_path, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

for i, r in enumerate(loaded):
    print("\n" + "#" * 72)
    print(f"# Example {i+1} / {len(loaded)} — {r.get('topic', '')}")
    print("#" * 72)
    print(f"\nPresent: {r.get('present_perspectives', [])}")
    rc = r.get('retrieved_context') or ''
    if rc.strip():
        prev = rc if len(rc) <= 2500 else rc[:2500] + "\n... [truncated]"
        print(f"\n--- Retrieved context ---\n{prev}")
    print(f"\n--- Ground truth ---\n{r.get('ground_truth', '')}")
    print(f"\n--- Model prediction ---\n{r.get('model_prediction', '')}")
    print("\n" + "-" * 72)

print(f"\nPrinted {len(loaded)} results from {eval_path}")


In [35]:
# Save model + checkpoint info to Drive
model_info = {
    "job_id": JOB_ID,
    "fine_tuned_model": FINE_TUNED_MODEL,
    "best_checkpoint": BEST_CHECKPOINT,
    "checkpoint_metrics": CHECKPOINT_METRICS,
    "train_file_id": TRAIN_FILE_ID,
    "validation_file_id": VAL_FILE_ID,
    "n_epochs": 3,
}

import json
with open(f"{DATA_REVISED_DIR}/model_info.json", 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print("Model info saved to Drive.")
print(f"Model ID used for inference: {FINE_TUNED_MODEL}")


✅ Model info saved to Drive!
Model ID: ft:gpt-4o-mini-2024-07-18:duke-trust-lab::DQdbeckt


In [38]:
# Cell 18: Print one evaluation example side by side for manual review

idx = 0  # change this to review different examples (0-9)

result = evaluation_results[idx]
print(f"Topic: {result['topic']}")
print(f"\n{'='*60}")
print("GROUND TRUTH OUTPUT:")
print(f"{'='*60}")
print(result['ground_truth'])
print(f"\n{'='*60}")
print("FINE-TUNED MODEL PREDICTION:")
print(f"{'='*60}")
print(result['model_prediction'])

Topic: The ethics of using AI to detect depression from social media activity

GROUND TRUTH OUTPUT:
**Missing perspective from engineering:**
Key insight: The design and implementation of AI systems for mental health must consider the engineering principles behind algorithm development and user interaction to ensure efficacy and ethical deployment.

Contribution: Engineering disciplines can contribute significantly to the conversation by focusing on the technical standards and frameworks required for the responsible development of AI tools in mental health contexts. Engineers can emphasize the importance of creating transparent algorithms that allow for interpretability, making it clear how AI systems derive their conclusions from user data. This transparency is crucial not only for the ethical use of data but also for fostering trust among users, especially when the technology interacts with sensitive mental health conditions like depression.

Moreover, engineers can advocate for the 

Save

In [39]:
# Save complete training record including RAG context
complete_record = {
    "metadata": {
        "model": "gpt-4o-mini-2024-07-18",
        "n_epochs": 3,
        "training_examples": len(training_examples),
        "holdout_examples": len(holdout_examples),
        "rag_domains": sorted(RAG_DOMAINS),
        "aux_domains": AUX_DOMAINS,
        "core_domains": CORE_DOMAINS,
        "suggestive_domains": SUGGESTIVE_DOMAINS,
    },
    "training_examples": training_examples,
}

with open(f"{DATA_REVISED_DIR}/complete_training_record.json", 'w', encoding='utf-8') as f:
    json.dump(complete_record, f, indent=2, ensure_ascii=False)

print("Complete training record saved!")


✅ Complete training record saved!


In [40]:
# Save training loss to Drive
loss_records = []
events = oai_client.fine_tuning.jobs.list_events(JOB_ID, limit=200)

for event in reversed(events.data):
    if event.data and 'train_loss' in event.data:
        loss_records.append({
            "step": event.data.get('step'),
            "train_loss": event.data.get('train_loss'),
            "valid_loss": event.data.get('valid_loss', None)
        })

with open(f"{DATA_REVISED_DIR}/training_loss.json", 'w') as f:
    json.dump(loss_records, f, indent=2)

print(f"✅ Training loss saved! {len(loss_records)} steps recorded.")

✅ Training loss saved! 120 steps recorded.
